In [6]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/merged_raw.csv', parse_dates=['InvoiceDate'])
print(df.shape)
df.head()

(1067371, 8)


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,CustomerID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [7]:
df['is_cancellation'] = df['Invoice'].str.startswith('C')
df['is_bad_price'] = df['Price'] <= 0
df['is_non_product'] = df['StockCode'].isin(
    ['POST','DOT','M','C2','D','S','BANK CHARGES','ADJUST','AMAZONFEE','PADS']
)
df['is_missing_customer'] = df['CustomerID'].isnull()

print(df[['is_cancellation','is_bad_price','is_non_product','is_missing_customer']].sum())

is_cancellation         19494
is_bad_price             6207
is_non_product           5783
is_missing_customer    243007
dtype: int64


In [8]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Dropped {before - after} duplicate rows. New shape: {df.shape}")

Dropped 34335 duplicate rows. New shape: (1033036, 12)


In [9]:
segmentation_df = df[
    (~df.is_cancellation) & (~df.is_bad_price) & (~df.is_non_product) & (~df.is_missing_customer)
].copy()

forecasting_df = df[
    (~df.is_bad_price) & (~df.is_non_product)
].copy()

recommendation_df = segmentation_df.copy()

print("Segmentation:", segmentation_df.shape)
print("Forecasting:", forecasting_df.shape)
print("Recommendation:", recommendation_df.shape)

Segmentation: (776592, 12)
Forecasting: (1021371, 12)
Recommendation: (776592, 12)


In [10]:
segmentation_df.to_csv('../data/segmentation_ready.csv', index=False)
forecasting_df.to_csv('../data/forecasting_ready.csv', index=False)
recommendation_df.to_csv('../data/recommendation_ready.csv', index=False)
print("Saved.")

Saved.
